# 05 — Comparación integral de modelos

Juntamos **todos** los modelos: los baselines (01), las tres familias tuneadas
(02–04: lineal, XGBoost, red neuronal) y los **ensembles adicionales** —
Random Forest, HistGradientBoosting y un **stacking** (XGBoost + MLP con
meta-modelo Ridge). Todos entrenados sobre el mismo train y evaluados sobre el
mismo test (≥2021), con los hiperparámetros **re-tuneados por CV temporal
honesta** (`_retune_all.py` → `retuning_cv_honesta.json`).

Comparamos de las tres formas que importan para decidir el mejor:
1. **Métricas de error** en test: RMSE (principal), MAE, R², sMAPE.
2. **Skill score** vs. la climatología (media por depto): cuánto aporta el clima
   del año por encima de "cada depto rinde lo de siempre".
3. **Test de Diebold–Mariano**: ¿la diferencia entre modelos es *significativa* o
   ruido? (H0: misma precisión; p<0.05 ⇒ diferencia real).

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))          # componente_b/ (datos, evaluacion)
warnings.filterwarnings('ignore')                  # silenciar ConvergenceWarning de sklearn

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

import datos, evaluacion as ev
from modelos import (LinearRegressor, XGBoostRegressor, NeuralNetRegressor,
                     RandomForestRegressorModel, HistGBMRegressor, StackingRegressorModel)

# Cultivo del estudio (cambiar a 'maiz' para reproducir con maíz).
CULTIVO = 'soja'
ds = datos.prepare(CULTIVO, use_agro=True, enc_smooth=10.0)
print(f'{CULTIVO}: {len(ds.feature_cols)} features | '
      f'train {ds.X_train.shape[0]} filas (≤{datos.TRAIN_END}) | '
      f'test {ds.X_test.shape[0]} filas (≥{datos.TEST_START})')


In [ ]:
import json
_bpath = 'retuning_cv_honesta.json'
BEST_ALL = json.load(open(_bpath, encoding='utf-8'))[CULTIVO] if os.path.exists(_bpath) else {}
def best_of(name, fallback):
    """best-params re-tuneados del modelo `name` (o `fallback` si no hay json)."""
    return BEST_ALL.get(name, {}).get('best_params', fallback)
print('re-tuning disponible:', sorted(k for k in BEST_ALL if not k.startswith('_')) or 'NO (usando fallbacks)')


## Entrenamiento de todos los modelos

Cada modelo con sus hiperparámetros re-tuneados (o un fallback razonable si no
está el json). El stacking recibe los años para armar sus meta-features
out-of-fold de forma temporal (sin leakage).

In [ ]:
yrs = ds.meta_train['campania_inicio'].values
def fit_pred(model):
    return model.fit(ds.X_train, ds.y_train).predict(ds.X_test)

preds = {}
preds['media global']  = ev.pred_media(ds)
preds['media x depto'] = ev.pred_media_depto(ds)
preds['Lineal']       = fit_pred(LinearRegressor(**best_of('linear', {'penalty':'ridge','alpha':10.0})))
preds['Random Forest'] = fit_pred(RandomForestRegressorModel(**best_of('rf', {}), random_state=42, n_jobs=-1))
preds['HistGBM']       = fit_pred(HistGBMRegressor(**best_of('hist_gbm', {}), random_state=42))
preds['XGBoost']       = fit_pred(XGBoostRegressor(**best_of('xgb', {}), random_state=42))
preds['Red neuronal']  = fit_pred(NeuralNetRegressor(**best_of('nn', {'hidden_dims':(64,32),'dropout':0.3}),
                                                     max_epochs=250, patience=30, random_state=42))
_stk = StackingRegressorModel(**best_of('stacking', {}))
_stk.fit(ds.X_train, ds.y_train, years=yrs)
preds['Stacking']      = _stk.predict(ds.X_test)

filas = [ev.evaluar(k, v, ds) for k, v in preds.items()]
tabla = ev.tabla_comparativa(filas, ordenar_por='rmse')
tabla

## Skill score vs. climatología (media por depto)

Positivo = el modelo le gana a la climatología; 0 = empata.

In [ ]:
ref = ev.pred_media_depto(ds)
skill = {k: ev.skill_score(ds.y_test, v, ref) for k, v in preds.items() if k != 'media x depto'}
skill = pd.Series(skill).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(skill.index[::-1], skill.values[::-1],
        color=['#55A868' if v > 0 else '#C44E52' for v in skill.values[::-1]])
ax.axvline(0, color='0.5', lw=1); ax.set_xlabel('skill score (1 - RMSE/RMSE_clima)')
ax.set_title('Skill vs. climatología por modelo'); plt.tight_layout(); plt.show()
skill.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
ev.plot_comparativa(tabla, metrica='rmse', ax=axes[0])
ev.plot_comparativa(tabla, metrica='r2', ax=axes[1])
plt.tight_layout(); plt.show()

## Test de Diebold–Mariano: ¿las diferencias son significativas?

Comparamos el **mejor modelo** (menor RMSE) contra cada uno de los demás. `dm<0`
= el mejor pierde *menos*; `p<0.05` = la diferencia es estadísticamente
significativa (no es ruido de muestreo del test).

In [ ]:
mejor = tabla.iloc[0]['modelo']
print('Mejor modelo por RMSE:', mejor)
filas_dm = []
for k in preds:
    if k == mejor: continue
    dm = ev.diebold_mariano(ds.y_test, preds[mejor], preds[k], loss='se')
    filas_dm.append({'vs': k, 'DM': dm['dm'], 'p_value': dm['p_value'],
                     'significativo (p<0.05)': dm['p_value'] < 0.05})
pd.DataFrame(filas_dm).sort_values('p_value')

## Predicho vs. real de los mejores modelos

In [ ]:
top3 = [m for m in tabla['modelo'] if m not in ('media global', 'media x depto')][:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, nombre in zip(axes, top3):
    ev.plot_pred_vs_real(ds.y_test, preds[nombre], nombre, color=ev.C_XGB, ax=ax)
plt.tight_layout(); plt.show()

## Conclusión

- Todos los modelos con clima **superan a la media global**; el listón real es la
  **media por departamento** (estructura espacial pura), y el **skill score** mide
  cuánto agrega el clima del año sobre eso.
- Los **ensembles de árboles** (Random Forest / HistGBM / XGBoost) dominan: capturan
  interacciones no lineales entre clima, espacio y tendencia. El **Diebold–Mariano**
  dice si la diferencia entre el puntero y el resto es real o ruido.
- El margen sobre el baseline por depto cuantifica cuánto aporta el **clima del año**
  por encima de "cada depto rinde lo de siempre" — lo difícil de predecir del rinde.

## ¿Por qué el R² es "bajo" (~0.25–0.30)?

No es (solo) que falte tunear: hay un **techo estructural** en predecir rinde de
clima a este nivel de agregación.

1. **La varianza reducible es chica.** El grueso del rinde es *espacial* (qué depto)
   y *tendencia* (qué década) — cosas que los baselines ya capturan. Lo que queda por
   explicar con el **clima del año** es una porción menor y ruidosa, así que aunque
   el modelo la capture bien, el R² total sube poco.
2. **Agregación depto-campaña.** Cada fila promedia miles de lotes con siembras,
   cultivares, suelos y manejo distintos: la relación clima→rinde de lote se diluye.
3. **Variables no observadas.** Manejo (fertilización, fecha de siembra, genética),
   plagas/enfermedades, granizo, y ruido de reporte de MAGYP no están en las features.
   El Componente A ya mostró que ~2/3 de las anomalías de rinde **no tienen firma
   climática**.
4. **Extrapolación temporal.** El test (2021–2024) cae *fuera* del rango de train: la
   tendencia y el régimen climático se corren. Se ve en que la feature `year`
   extrapola (ayuda al lineal, la cortan los árboles) y en que el `es_anomalo` del
   VAE marca casi todo el test (el score sube por *distribution shift*, no por
   anomalía real).

**Sobre el latente del Componente A** (nbs 02–04): concatenar el **latente del VAE**
suele empeorar la regresión (y solo-latente es lo peor): ese latente resume el clima
*normalizado por depto*, así que tira la señal espacial y de tendencia que es justo la
que más predice. `es_anomalo` tampoco ayuda, dominado por el *distribution shift*.
Coherente con el techo del Componente A.

## ¿Qué podemos modificar para mejorar?

En orden aproximado de impacto esperado:

1. **Cambiar el target a algo aprendible.** En vez del rinde absoluto, predecir la
   **anomalía de rinde** (residuo sobre la media/tendencia por depto, p. ej. el
   `z_rinde` del Componente A). Saca la parte "fácil" (espacio+tendencia) y deja que
   el modelo se concentre en la señal climática — R² más honesto de lo que sí se
   puede predecir.
2. **Features agronómicas, no promedios mensuales crudos.** Índices en la **ventana
   crítica** por cultivo: balance hídrico acumulado, días de estrés térmico (Tmax>32
   en floración), rachas secas, grados-día. El Componente A ya tiene `add_agro_features`.
3. **NDVI como predictor directo, no como una feature más.** El NDVI de
   floración/llenado es un proxy casi directo del rinde; conviene usarlo con más peso
   (o un modelo aparte) en vez de mezclarlo entre 60+ columnas.
4. **Tratar la extrapolación temporal.** Detrendear el rinde antes de modelar, o usar
   validación que imite el gap train→test; para los árboles, no darles `year` crudo.
5. **Modelo residual / híbrido.** Baseline fuerte = media por depto (+ tendencia), y
   un modelo que aprenda **solo el residuo** con el clima. Suele ganarle a predecir
   el absoluto de una.
6. **Robustez al ruido de etiqueta.** Pérdida robusta (Huber/cuantil) por el ruido de
   reporte de MAGYP, y quizás modelar por región/cultivo por separado.

La limitación de fondo (agregación depto-campaña + variables no observadas) solo se
levanta con **datos más finos** (lote/píxel, manejo), que exceden este panel.

Para reproducir con maíz: cambiar `CULTIVO = 'maiz'` en la celda de setup y
re-ejecutar los notebooks.